# From Tidy to Wide Data

DS 2023 | Communicating with Data

We have seen that a tidy table is one in which:

- Each **row** stands for a unique **observation**. The observation is the **subject** of the recorded fact.
- Each **column** stands for a **variable**. The variable is the **predicate** of the recorded fact.
- Each **cell** possesses the **value** of that variable for a given observation. The value is the **object** (so-called) of the recorded fact.

## The complete data set

Let's explore the properties of tidy data in more depth using the complete data set of health outcomes from countries over years.

We will import the Health Expenditure and Life Expectancy `healthexp` data set from Seaborn's collection.

> [!note] This data set represents the annual per-capita health-care spending paired with life expectancy at birth for six wealthy nations for the years 1970 to 2020. It is sourced from the [Our World in Data](https://ourworldindata.org/) website. It combines OECD health-expenditure statistics (per-capita spending) with UN World Population Prospects (life expectancy).

In [1]:
import pandas as pd
import seaborn as sns

healthexp = sns.load_dataset("healthexp")
healthexp.index.name = 'obs_id'
healthexp.info()
healthexp

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274 entries, 0 to 273
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             274 non-null    int64  
 1   Country          274 non-null    object 
 2   Spending_USD     274 non-null    float64
 3   Life_Expectancy  274 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 8.7+ KB


,Year,Country,Spending_USD,Life_Expectancy
obs_id,,,,
0,1970,Germany,252.311,70.6
1,1970,France,192.143,72.2
2,1970,Great Britain,123.993,71.9
3,1970,Japan,150.437,72.0
4,1970,USA,326.961,70.9
...,...,...,...,...
269,2020,Germany,6938.983,81.1
270,2020,France,5468.418,82.3
271,2020,Great Britain,5018.700,80.4


## Identify columns uses

In [2]:
group_cols = ['Country','Year']
measure_cols = ['Spending_USD','Life_Expectancy']

## Create a wide table

### Move 1: Use the group columns to define the index

If the group columns form a unique key -- and here they do -- then can replace the index with them.

In [3]:
healthexp1 = healthexp.set_index(group_cols)
healthexp1

,,Spending_USD,Life_Expectancy
Country,Year,,
Germany,1970,252.311,70.6
France,1970,192.143,72.2
Great Britain,1970,123.993,71.9
Japan,1970,150.437,72.0
USA,1970,326.961,70.9
...,...,...,...
Germany,2020,6938.983,81.1
France,2020,5468.418,82.3
Great Britain,2020,5018.700,80.4


> [!note] An index with two columns is called a **MultiIndex** in Pandas. Indexes can have many columns.
>
> When selecting rows from a dataframe with a MultiIndex, use a tuple to select more than one column.
>
> For example, to select the row for Germany in 1970, do this:
> ```python
> healthexp1.loc[('Germany', 1979)]
> ```

### Move 2: Unstack on a measurement variable

Here is how you unstack on **Spending_USD**.

First, notice what happens when we select a single column.

In [4]:
healthexp1['Spending_USD'] # Selecting one column produces a Series with a MultiIndex

,,Spending_USD
Country,Year,
Germany,1970,252.311
France,1970,192.143
Great Britain,1970,123.993
Japan,1970,150.437
USA,1970,326.961
...,...,...
Germany,2020,6938.983
France,2020,5468.418
Great Britain,2020,5018.700


When we unstack the series, we get a wide dataframe.

In [8]:
healthexp1['Spending_USD'].unstack().T.head(10) # Unstacking projects the second index column onto axis 1

Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884


Similary, we can unstack on **Life_Expectancy**:

In [6]:
healthexp1['Life_Expectancy'].unstack().T.head(10)

Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,72.2,70.6,71.9,72.0,70.9
1971,72.8,NaN,70.8,71.9,72.9,71.2
1972,NaN,NaN,71.0,NaN,73.2,71.2
1973,NaN,NaN,71.3,NaN,73.4,71.4
1974,NaN,NaN,71.5,NaN,73.7,72.0
1975,NaN,73.0,71.4,NaN,74.3,72.7
1976,73.8,NaN,71.8,NaN,74.8,72.9
1977,NaN,NaN,72.5,NaN,75.3,73.3
1978,NaN,NaN,72.4,NaN,75.7,73.5


### Or just use pivot

We can achieve the same effect using `pivot`:

In [7]:
healthexp_spending = healthexp.pivot(index='Country', columns='Year', values='Spending_USD')
healthexp_spending.T.head(10)

Country,Canada,France,Germany,Great Britain,Japan,USA
Year,,,,,,
1970,NaN,192.143,252.311,123.993,150.437,326.961
1971,313.391,NaN,298.251,134.172,163.854,357.988
1972,NaN,NaN,337.364,NaN,185.390,397.097
1973,NaN,NaN,384.541,NaN,205.778,439.302
1974,NaN,NaN,452.744,NaN,242.018,495.114
1975,NaN,363.610,532.481,NaN,284.269,560.750
1976,543.337,NaN,591.098,NaN,303.725,638.851
1977,NaN,NaN,647.352,NaN,340.628,726.241
1978,NaN,NaN,729.457,NaN,392.577,808.884


> [!tip] The choice of when to use pivot versus unstack is usually a matter or workflow. Unstacking is natural when you are working with tables that have an index of two or more columns, whereas pivots are a good choice to go directly to a wide table.

## Question

What do we notice about these tables? Are they tidy?

```{dropdown} Answer
No, they are not, because rows are not observations and columns are not variables.

Instead, rows and columns are both variables, with each domain projected onto their respective axes.

Also, the content of the table &mdash; the cells &mdash; are all values from the same measurement variable.
```